In [0]:
%fs ls "/Volumes/workspace/default/covid1/newdf1_csv/"

In [0]:
%fs ls "dbfs:/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-06-2025.json"

In [0]:
df = spark.read.option('multiline','true').option('multiline',True).json("dbfs:/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-06-2025.json")
display(df)

In [0]:
# from pyspark.sql.functions import col, explode, sum as spark_sum, lit
# # Explode card payments
# df_payment_card = df.select(
#     explode(col("payments.cards")).alias("card")
# )

# #Explode cash payment
# df_payment_cash = df.select(
#     explode(col("payments.cash")).alias("cash_amount")
# )

# # Summarize card totals by cardType
# df_card_sum = (
#     df_payment_card
#     .filter(col("card.totalAmount").cast("double") > 0)
#     .groupby(col("card.cardType").alias("payment_type"))
#     .agg(
#         spark_sum(col("card.totalAmount").cast("double")).alias("totalAmount")
#     )
# )

# #cash total
# df_cash_sum = (
#     df_payment_cash
#     .filter(col("cash_amount").cast("double") > 0)
#     .agg(
#         spark_sum(col("cash_amount").cast("double")).alias("totalAmount")
#     )
#     .withColumn("payment_type", lit("cash"))
# )

# #. Combine card + cash
# df_ans = df_card_sum.unionByName(df_cash_sum)

# deployment_name=
# # deployment columns
# df_final = (
#     df_ans
#     .withColumn("deployment_name", lit("ALBAIK-SQ DB04 - AJMAN CITY CENTRE"))
#     .withColumn("date", lit("2025-09-05"))
#     .select("deployment_name", "date", "payment_type", "totalAmount")
# )

# df_final.display()


In [0]:
dbutils.widgets.dropdown(
    "stores_dropdown",
    "ALBAIK - SQ DB04 - AJMAN CITY CENTRE",
    [
        "ALBAIK - SQ DB04 - AJMAN CITY CENTRE",
        "ALBAIK - SQ DB05 - AL MAJAZ",
    ],
    "Select Store"
)
dbutils.widgets.dropdown(
    "Select_Date",
    "2025-09-06",
    ["2025-09-06", "2025-09-07", "2025-09-08"],
    "Select Date"
)
selected_store = dbutils.widgets.get("stores_dropdown")
selected_date = dbutils.widgets.get("Select_Date")

print("Selected Store:", selected_store)
print("Selected Date:", selected_date)
store_file_map = {
    "ALBAIK - SQ DB04 - AJMAN CITY CENTRE": {
        "2025-09-06": "/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-06-2025.json",
        "2025-09-07": "/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-07-2025.json",
        "2025-09-08": "/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-08-2025.json"
    },
    "ALBAIK - SQ DB05 - AL MAJAZ": {
        "2025-09-06": "/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB05 - AL MAJAZ - 100500509-06-2025.json",
        "2025-09-07": "/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB05 - AL MAJAZ - 100500509-07-2025.json",
        "2025-09-08": "/Volumes/workspace/default/covid1/newdf1_csv/ALBAIK - SQ DB05 - AL MAJAZ - 100500509-08-2025.json"
    }
}




In [0]:
file_path = store_file_map[selected_store][selected_date]
print("Reading file:", file_path)

# Load JSON
df1 = spark.read.json(file_path, multiLine=True)
display(df1)

In [0]:
from pyspark.sql.functions import col, explode, sum as spark_sum, lit
# Explode card payments
df_payment_card = df1.select(
    explode(col("payments.cards")).alias("card")
)

#Explode cash payment
df_payment_cash = df1.select(
    explode(col("payments.cash")).alias("cash_amount")
)

# Summarize card totals by cardType
df_card_sum = (
    df_payment_card
    .filter(col("card.totalAmount").cast("double") > 0)
    .groupby(col("card.cardType").alias("payment_type"))
    .agg(
        spark_sum(col("card.totalAmount").cast("double")).alias("totalAmount")
    )
)

#cash total
df_cash_sum = (
    df_payment_cash
    .filter(col("cash_amount").cast("double") > 0)
    .agg(
        spark_sum(col("cash_amount").cast("double")).alias("totalAmount")
    )
    .withColumn("payment_type", lit("cash"))
)

#. Combine card + cash
df_ans = df_card_sum.unionByName(df_cash_sum)
""
deployment_name=dbutils.widgets.get("stores_dropdown")
date=dbutils.widgets.get("Select_Date")
# deployment columns
df_final = (
    df_ans
    .withColumn("deployment_name", lit(deployment_name))
        .withColumn("date", lit(date))
    .select("deployment_name", "date", *df_ans.columns)
)

df_final.display()
